# MyMLTool — Usage Demo

This notebook shows how to use the two reusable modules:

1. **`src.data_prep`** — the data-preparation pipeline (load a CSV, split features/target, train/test split, optional feature scaling).
2. **`src.feature_shift`** — the `FeatureShiftDetector` that flags distribution drift in a serving model's incoming data versus its training reference.

Everything below uses synthetic data so the notebook runs end-to-end with no external files.

## 0. Setup

Make the project importable regardless of where the notebook is launched from (it searches upward for the repo root), then import the public API.

In [1]:
import os
import sys
from pathlib import Path


def find_repo_root(start):
    """Walk up from `start` until we find the repo (src/data_prep.py)."""
    here = Path(start).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "data_prep.py").exists():
            return candidate
    raise RuntimeError("Could not locate the repo root")


REPO_ROOT = find_repo_root(os.getcwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)

Repo root: /mnt/d/IT/githubProbject/MyMLTool


In [2]:
import json

import numpy as np
import pandas as pd

from src.data_prep import (
    load_dataset,
    split_features_target,
    split_train_test,
    scale_features,
    prepare_data,
)
from src.feature_shift import FeatureShiftDetector, compute_psi, ks_drift, chi2_drift

## 1. Create a synthetic dataset

We build a small customer dataset with two numeric features (`age`, `salary`), one categorical feature (`region`), and a binary target (`purchased`) in the **last column** — the convention the pipeline expects. We save it to a CSV under `output/` (git-ignored) so we can demo loading from disk.

In [3]:
rng = np.random.default_rng(42)
N = 1000

salary = rng.normal(60000, 10000, N)
training_df = pd.DataFrame(
    {
        "age": rng.normal(40, 5, N),
        "salary": salary,
        "region": rng.choice(["north", "south", "east"], N, p=[0.5, 0.3, 0.2]),
        "purchased": (salary > 60000).astype(int),  # target in the LAST column
    }
)

output_dir = REPO_ROOT / "output"
output_dir.mkdir(exist_ok=True)
csv_path = output_dir / "demo_training_data.csv"
training_df.to_csv(csv_path, index=False)

print(f"Wrote {len(training_df)} rows to {csv_path}")
training_df.head()

Wrote 1000 rows to /mnt/d/IT/githubProbject/MyMLTool/output/demo_training_data.csv


,age,salary,region,purchased
0,39.703587,63047.170798,north,1
1,36.353565,49600.158938,north,0
2,37.927635,67504.511958,south,1
3,43.169552,69405.647164,north,1
4,40.014966,40489.648113,north,0


## 2. Data preprocessing — step by step

The pipeline exposes small, single-purpose functions so you can use exactly the steps you need.

### 2.1 `load_dataset` — read a CSV into a DataFrame

Raises `FileNotFoundError` for a missing path and `ValueError` for an empty file.

In [4]:
df = load_dataset(str(csv_path))
print("Shape:", df.shape)
df.dtypes

Shape: (1000, 4)


age          float64
salary       float64
region           str
purchased      int64
dtype: object

### 2.2 `split_features_target` — separate X and y

By default the **last column** is the target (matching the original scripts' `iloc[:, :-1]` / `iloc[:, -1]`). You can also pass a column name or integer position via `target_col`.

In [5]:
X, y, feature_names = split_features_target(df)  # target_col=-1 by default
print("Feature names:", feature_names)
print("X shape:", X.shape, "| y shape:", y.shape)

# Equivalent, selecting the target explicitly by name:
X2, y2, _ = split_features_target(df, target_col="purchased")
print("Same target selected by name:", np.array_equal(y, y2))

Feature names: ['age', 'salary', 'region']
X shape: (1000, 3) | y shape: (1000,)
Same target selected by name: True


### 2.3 `split_train_test` — train/test split

A thin wrapper over scikit-learn's `train_test_split` that keeps the repo's defaults (`test_size=0.2`, `random_state=0`).

In [6]:
X_train, X_test, y_train, y_test = split_train_test(X, y)
print("Train rows:", X_train.shape[0], "| Test rows:", X_test.shape[0])

Train rows: 800 | Test rows: 200


### 2.4 `scale_features` — standardize numeric features

Fits a `StandardScaler` on the training split and transforms both splits. Returns the fitted scaler so you can apply the **same** transform to future serving data. (Scaling is for numeric features — we use a numeric-only frame here.)

In [7]:
numeric_df = df[["age", "salary", "purchased"]]
Xn, yn, _ = split_features_target(numeric_df)
Xn_train, Xn_test, yn_train, yn_test = split_train_test(Xn, yn)

Xn_train_scaled, Xn_test_scaled, scaler = scale_features(Xn_train, Xn_test)
print("Train mean after scaling (~0):", np.round(Xn_train_scaled.mean(axis=0), 6))
print("Train std  after scaling (~1):", np.round(Xn_train_scaled.std(axis=0), 6))
print("Fitted scaler:", scaler)

Train mean after scaling (~0): [ 0. -0.]
Train std  after scaling (~1): [1. 1.]
Fitted scaler: StandardScaler()


### 2.5 `prepare_data` — the whole pipeline in one call

Runs load → split features/target → train/test split → optional scaling, and returns a `PreparedData` dataclass bundling everything (including `feature_names` and the fitted `scaler`).

In [8]:
# Numeric-only CSV so scale=True can standardize every feature.
numeric_csv = output_dir / "demo_numeric_data.csv"
numeric_df.to_csv(numeric_csv, index=False)

prepared = prepare_data(str(numeric_csv), target_col="purchased", scale=True)

print("feature_names :", prepared.feature_names)
print("X_train shape :", prepared.X_train.shape)
print("X_test shape  :", prepared.X_test.shape)
print("scaler        :", prepared.scaler)

feature_names : ['age', 'salary']
X_train shape : (800, 2)
X_test shape  : (200, 2)
scaler        : StandardScaler()


### 2.6 Train a model on the prepared data

`prepare_data` output drops straight into any scikit-learn estimator.

In [9]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=0)
model.fit(prepared.X_train, prepared.y_train.astype(int))
accuracy = model.score(prepared.X_test, prepared.y_test.astype(int))
print(f"Test accuracy: {accuracy:.3f}")

Test accuracy: 0.980


## 3. Feature-shift detection

At serving time you want to know whether incoming data still looks like the data the model was trained on. `FeatureShiftDetector` is fit on the **training reference** (features only, no target) and then scores each incoming batch.

Drift signals per feature:

| Feature type | Statistical test | Magnitude metric |
|---|---|---|
| Numerical | Kolmogorov–Smirnov (`ks_2samp`) | PSI |
| Categorical | Chi-square (`chi2_contingency`) | PSI |

A feature is flagged as drifted if the test p-value < `alpha` (0.05) **or** PSI ≥ `psi_threshold` (0.25).

### 3.1 Fit the detector on the training reference

In [10]:
reference = training_df.drop(columns=["purchased"])  # features only
detector = FeatureShiftDetector(reference).fit()

print("Numerical features :", detector.numerical_features)
print("Categorical features:", detector.categorical_features)

Numerical features : ['age', 'salary']
Categorical features: ['region']


### 3.2 A healthy batch (same distribution) — no drift

Draw a fresh batch from the **same** distribution as training. The detector should report `drifted = False`.

In [11]:
rng2 = np.random.default_rng(7)
healthy_batch = pd.DataFrame(
    {
        "age": rng2.normal(40, 5, 500),
        "salary": rng2.normal(60000, 10000, 500),
        "region": rng2.choice(["north", "south", "east"], 500, p=[0.5, 0.3, 0.2]),
    }
)

report = detector.detect(healthy_batch)
print("Drift detected? ", report.drifted)
print("Drifted features:", report.drifted_features)

Drift detected?  False
Drifted features: []


### 3.3 A drifted batch — drift detected

Now simulate a runtime batch where the population has shifted: older customers, higher salaries, and a different regional mix.

In [12]:
rng3 = np.random.default_rng(99)
drifted_batch = pd.DataFrame(
    {
        "age": rng3.normal(65, 5, 500),       # shifted up
        "salary": rng3.normal(95000, 10000, 500),  # shifted up
        "region": rng3.choice(["north", "south", "east"], 500, p=[0.1, 0.2, 0.7]),  # shifted mix
    }
)

report = detector.detect(drifted_batch)
print("Drift detected? ", report.drifted)
print("Drifted features:", report.drifted_features)

Drift detected?  True
Drifted features: ['age', 'salary', 'region']


### 3.4 Inspect the per-feature report

Each `DriftReport` carries a list of per-feature results with the test used, the statistic, the p-value, the PSI, and the drift flag. It's easy to turn into a DataFrame for monitoring dashboards.

In [13]:
report_df = pd.DataFrame([r.__dict__ for r in report.results])
report_df

,feature,type,test,statistic,p_value,psi,drifted
0,age,numerical,ks,0.986000,3.952525e-323,12.433856,True
1,salary,numerical,ks,0.935000,1.877449e-322,10.804488,True
2,region,categorical,chi2,386.345306,1.276976e-84,1.299541,True


### 3.5 Serialize the report

`DriftReport.to_dict()` produces a JSON-serializable payload you can log or ship to a monitoring service.

In [14]:
payload = report.to_dict()
print(json.dumps(payload, indent=2)[:600], "...")

{
  "drifted": true,
  "drifted_features": [
    "age",
    "salary",
    "region"
  ],
  "n_features": 3,
  "results": [
    {
      "feature": "age",
      "type": "numerical",
      "test": "ks",
      "statistic": 0.986,
      "p_value": 4e-323,
      "psi": 12.433855885838662,
      "drifted": true
    },
    {
      "feature": "salary",
      "type": "numerical",
      "test": "ks",
      "statistic": 0.935,
      "p_value": 1.9e-322,
      "psi": 10.804488485197837,
      "drifted": true
    },
    {
      "feature": "region",
      "type": "categorical",
      "test": "chi2",
      "st ...


### 3.6 Using the low-level drift functions directly

If you only need a single signal, the building blocks are public too.

In [15]:
ref_age = reference["age"].values
print("KS, same dist  :", ks_drift(ref_age, healthy_batch["age"].values))
print("KS, shifted    :", ks_drift(ref_age, drifted_batch["age"].values))
print("PSI, same dist :", round(compute_psi(ref_age, healthy_batch["age"].values), 4))
print("PSI, shifted   :", round(compute_psi(ref_age, drifted_batch["age"].values), 4))
print("Chi-square region, shifted:", chi2_drift(reference["region"], drifted_batch["region"]))

KS, same dist  : {'statistic': 0.065, 'p_value': 0.11787377488127634, 'drifted': False}
KS, shifted    : {'statistic': 0.986, 'p_value': 4e-323, 'drifted': True}
PSI, same dist : 0.0328
PSI, shifted   : 12.4339
Chi-square region, shifted: {'statistic': 386.34530591175775, 'p_value': 1.276976092958377e-84, 'drifted': True}


## Summary

- **`prepare_data(csv_path, target_col=-1, test_size=0.2, random_state=0, scale=False)`** gives you a ready-to-train `PreparedData` bundle in one call — or compose `load_dataset` / `split_features_target` / `split_train_test` / `scale_features` yourself.
- **`FeatureShiftDetector(reference_df).fit()`** then **`.detect(incoming_df)`** returns a `DriftReport` telling you whether — and where — your serving data has drifted from training.

See `docs/flow/system_flow.md` for the full architecture and `tests/` for more usage examples.